# Week 17, Agent Ops: Observability, Cost & Production Evals

```text
# Requirements: pip install numpy pandas
```

> No API key needed. Everything runs locally against seeded synthetic data.

Build the production layer an always-on agent needs: a **JSONL span/trace logger**, a **cost dashboard** from a synthetic usage log, a **production-eval sampling** demo, and a generated **ops runbook**, ending with a **coverage metric** that says how much of your traffic is actually instrumented.

## Why ops is part of the agent track

An agent is a *process*, not a single answer: when it fails, the failure lives in a specific step (a retrieval, a tool call, a handoff), and the only way to find it is the **trace**. Once the agent is always-on, it is also a **cost** and a **drift risk**, so you meter spend, and you keep grading a sample of live traffic against a golden set. Three ideas to fix: **traces** (structured spans per step), **cost dashboards** (spend per model/day), and **production-eval sampling** (grade live traffic on a cadence). The runbook ties them together; the coverage metric is the honest number for 'how much do we actually see?'

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
_root = pathlib.Path.cwd()
while not (_root / "zoro").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

import json, time, tempfile
import numpy as np

SEED = 42
print("imports ready")

## A lightweight span/trace logger (JSONL)

A **span** is one step (one model call, one tool call) with its inputs, outputs, latency, and token cost; a **trace** is the sequence of spans for one run. This logger writes one JSON object per span, no external platform, just a file you can `grep`.

In [ ]:
class SpanLogger:
    # Minimal JSONL span logger: one record per span, with ids to link spans into traces.
    def __init__(self, path):
        self.path = pathlib.Path(path)
        if self.path.exists():
            self.path.unlink()
        self._counter = 0

    def _emit(self, rec):
        with open(self.path, "a") as f:
            f.write(json.dumps(rec) + "\n")

    def new_span(self, name, trace_id=0, **attrs):
        self._counter += 1
        return _Span(self, self._counter, name, trace_id, attrs)

    def trace(self, name):
        # Decorator: wrap any function so each call emits a span.
        def deco(fn):
            def wrapper(*args, **kwargs):
                with self.new_span(name, fn=fn.__name__):
                    return fn(*args, **kwargs)
            return wrapper
        return deco

    def count(self):
        return self._counter

class _Span:
    def __init__(self, logger, span_id, name, trace_id, attrs):
        self.logger = logger
        self.span_id = span_id
        self.name = name
        self.trace_id = trace_id
        self.attrs = dict(attrs)

    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        latency = time.perf_counter() - self.t0
        self.logger._emit({
            "trace_id": self.trace_id,
            "span_id": self.span_id,
            "name": self.name,
            "attrs": self.attrs,
            "latency_ms": round(latency * 1000, 3),
            "tokens": int(self.attrs.get("tokens", 0)),
        })
        return False

logger = SpanLogger(pathlib.Path(tempfile.gettempdir()) / "zoro_w17_spans.jsonl")
print("SpanLogger ready ->", logger.path)

In [ ]:
# Wrap a few (fake) agent steps and run them; each call emits a span.
@logger.trace("retrieve")
def retrieve(query, tokens=120):
    time.sleep(0.001)
    return {"docs": ["doc-1", "doc-2"], "tokens": tokens}

@logger.trace("generate")
def generate(context, tokens=80):
    time.sleep(0.002)
    return "grounded answer using " + str(len(context)) + " docs"

for i in range(3):
    ctx = retrieve("shipment " + str(i))
    generate(ctx["docs"], tokens=60 + i * 10)

lines = logger.path.read_text().strip().splitlines()
print("spans written:", len(lines))
print("first 2 span records:")
for ln in lines[:2]:
    print(" ", ln)

## A synthetic usage log

Generate a seeded log of 200 calls across three models so the cost dashboard has something real to aggregate.

In [ ]:
def gen_usage(n=200, seed=123):
    rng = np.random.default_rng(seed)
    models = ["gpt-4o-mini", "gpt-4o", "hermes-4-14b"]
    probs = [0.5, 0.3, 0.2]
    rows = []
    for i in range(n):
        model = models[int(rng.choice(3, p=probs))]
        in_tok = int(rng.integers(200, 5000))
        out_tok = int(rng.integers(50, 800))
        latency = in_tok / 50.0 + out_tok / 30.0 + float(rng.normal(0, 0.2))
        day = int(rng.integers(0, 7))
        rows.append({"model": model, "day": day, "in_tokens": in_tok, "out_tokens": out_tok, "latency_ms": round(latency * 1000, 1)})
    return rows

usage = gen_usage(seed=123)
print("usage rows:", len(usage))
print("sample:", usage[0])

## Cost dashboard

Aggregate spend per model (and per day) from the usage log, then render a text dashboard with an ASCII bar chart. Prices are illustrative per 1M tokens; a local model (`hermes-4-14b`) costs $0 for tokens but you pay in hardware and latency.

In [ ]:
PRICES = {  # (input, output) USD per 1M tokens
    "gpt-4o-mini": (0.15, 0.60),
    "gpt-4o": (2.50, 10.00),
    "hermes-4-14b": (0.0, 0.0),
}

def cost_of(row):
    pin, pout = PRICES[row["model"]]
    return row["in_tokens"] / 1e6 * pin + row["out_tokens"] / 1e6 * pout

agg = {}
for r in usage:
    m = r["model"]
    a = agg.setdefault(m, {"cost": 0.0, "tokens": 0, "calls": 0, "latency": 0.0})
    a["cost"] += cost_of(r)
    a["tokens"] += r["in_tokens"] + r["out_tokens"]
    a["calls"] += 1
    a["latency"] += r["latency_ms"]

print("model           calls   tokens      cost($)   avg_latency(ms)")
for m, a in sorted(agg.items(), key=lambda kv: -kv[1]["cost"]):
    print("%-15s %5d   %8d   %8.4f   %12.1f" % (m, a["calls"], a["tokens"], a["cost"], a["latency"] / a["calls"]))

total_cost = sum(a["cost"] for a in agg.values())
print("TOTAL cost: $%.4f" % total_cost)

print()
print("--- cost share (ASCII) ---")
max_cost = max(a["cost"] for a in agg.values()) or 1.0
for m, a in sorted(agg.items(), key=lambda kv: -kv[1]["cost"]):
    bar = "#" * int(round(a["cost"] / max_cost * 40))
    print("%-15s %s $%.4f" % (m, bar, a["cost"]))

## Production-eval sampling

Grade a sample of live traffic against a threshold and estimate the pass rate with a confidence interval. This is the standing discipline from Week 11, applied to an always-on agent: sample → grade → decide, on a cadence.

In [ ]:
import math

def sample_eval(n=30, seed=7, true_pass=0.90):
    rng = np.random.default_rng(seed)
    return rng.random(n) < true_pass

samples = sample_eval(seed=7)
n = len(samples)
passed = int(samples.sum())
p = passed / n
se = math.sqrt(p * (1 - p) / n)
lo, hi = max(0.0, p - 1.96 * se), min(1.0, p + 1.96 * se)

THRESHOLD = 0.85
print("production sample: %d/%d passed" % (passed, n))
print("pass rate %.4f  (95%% CI: %.4f - %.4f)" % (p, lo, hi))
print("decision:", "SHIP (>= %.2f)" % THRESHOLD if p >= THRESHOLD else "HOLD (below %.2f) - investigate before shipping" % THRESHOLD)

## Ops runbook

Generate a runbook markdown from the measured numbers: thresholds, dashboards to watch, and what-to-do-when. Written to a temp file and previewed.

In [ ]:
runbook = (
    "# ZoroLab Agent Ops Runbook\n\n"
    "## Thresholds\n"
    "- Production pass rate >= 0.85 on a 30-trace daily sample (today: %.4f).\n"
    "- Cost/day alert at $2.00 (today's 200-call sample: $%.4f).\n"
    "- p95 latency alert at 1200 ms.\n"
    "- Coverage: target 100%% of agent calls instrumented (today: see final metric).\n\n"
    "## Dashboards to watch\n"
    "- Cost per model per day (text dashboard in this notebook).\n"
    "- Span latency per node; token spend per node.\n"
    "- Pass-rate trend over the last 7 samples.\n\n"
    "## What to do when\n"
    "1. Pass rate < threshold -> read the failing traces, cluster, fix the largest class, add it to the eval set.\n"
    "2. Cost spike -> check the most expensive model's call count; cache or downgrade the easy path.\n"
    "3. Latency spike -> find the slowest span; consider parallelization or a smaller model.\n"
    "4. Coverage < 1.0 -> route the uninstrumented call path through the tracer.\n"
) % (p, total_cost)

runbook_path = pathlib.Path(tempfile.gettempdir()) / "zoro_w17_runbook.md"
runbook_path.write_text(runbook)
print(runbook_path)
print("--- preview ---")
print(runbook)

In [ ]:
# Coverage metric: what fraction of agent operations are actually instrumented?
rng2 = np.random.default_rng(7)

@logger.trace("agent_step")
def traced_step(x):
    return x * 2

def untraced_step(x):
    return x * 2  # a call path that bypasses the tracer

total_ops = 100
instrumented = 0
for i in range(total_ops):
    if rng2.random() < 0.85:
        traced_step(i)
        instrumented += 1
    else:
        untraced_step(i)

coverage = round(instrumented / total_ops, 4)
print("instrumented", instrumented, "/", total_ops, "operations")
print("COVERAGE", coverage)